# 🌊 River Turbidity Detection - Model Training

**This Notebook**:
- Build Xception model architecture
- Implement 2-stage transfer learning
- Train with data augmentation
- Monitor with TensorBoard
- Save best model checkpoint

## Step 1: Import Libraries

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import Xception
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard, CSVLogger
from datetime import datetime
import logging

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## Step 2: Verify GPU and Define Hyperparameters

In [9]:
# Check GPU
gpus = tf.config.list_physical_devices('GPU')
print(f"🎮 GPU Configuration:")
print(f"   Available GPUs: {len(gpus)}")
if gpus:
    for gpu in gpus:
        print(f"   ✓ {gpu}")
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print("   ✅ Memory growth enabled")
else:
    print("   ⚠️  No GPU detected - using CPU (still works fine!)")
    print("   💡 To enable GPU: pip install tensorflow[and-cuda]")

# Check CPU
cpus = tf.config.list_physical_devices('CPU')
print(f"\n💻 CPU Configuration:")
print(f"   Available CPUs: {len(cpus)}")
if cpus:
    print(f"   ✓ CPU ready for training")

# Hyperparameters - Adjusted for CPU/GPU compatibility
BATCH_SIZE = 16  # Reduced from 32 for more stable training
EPOCHS_STAGE1 = 20  # Frozen base
EPOCHS_STAGE2 = 80  # Fine-tuning
LEARNING_RATE_STAGE1 = 0.001
LEARNING_RATE_STAGE2 = 0.0001
TARGET_SIZE = (299, 299)
NUM_CLASSES = 2

print(f"\n⚙️  Hyperparameters:")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Stage 1 Epochs: {EPOCHS_STAGE1} (frozen base)")
print(f"   Stage 2 Epochs: {EPOCHS_STAGE2} (fine-tuning)")
print(f"   Learning Rates: {LEARNING_RATE_STAGE1} → {LEARNING_RATE_STAGE2}")
print(f"   Image Size: {TARGET_SIZE}")
print(f"\n📌 Note: Batch size reduced to 16 for stability")

🎮 GPU Configuration:
   Available GPUs: 0
   ⚠️  No GPU detected - using CPU

⚙️  Hyperparameters:
   Batch Size: 32
   Stage 1 Epochs: 20 (frozen base)
   Stage 2 Epochs: 80 (fine-tuning)
   Learning Rates: 0.001 → 0.0001
   Image Size: (299, 299)


## Step 3: Setup Data Generators

In [3]:
# Training data generator with augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Validation and test data generators (no augmentation)
val_test_datagen = ImageDataGenerator(rescale=1./255)

# Load datasets
BASE_DIR = Path('.')
PROCESSED_DIR = BASE_DIR / 'data_processed'

train_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR / 'train',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_test_datagen.flow_from_directory(
    PROCESSED_DIR / 'val',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    PROCESSED_DIR / 'test',
    target_size=TARGET_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print("✅ Data generators created:")
print(f"   Train generator: {train_generator.samples} samples")
print(f"   Val generator: {val_generator.samples} samples")
print(f"   Test generator: {test_generator.samples} samples")
print(f"   Classes: {train_generator.class_indices}")

Found 142 images belonging to 2 classes.
Found 30 images belonging to 2 classes.
Found 32 images belonging to 2 classes.
✅ Data generators created:
   Train generator: 142 samples
   Val generator: 30 samples
   Test generator: 32 samples
   Classes: {'jernih': 0, 'keruh': 1}


## Step 4: Build Model Architecture

In [4]:
def create_model():
    """
    Create Xception model with custom classification head
    """
    # Load pre-trained Xception model
    base_model = Xception(
        input_shape=(299, 299, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model layers
    base_model.trainable = False
    
    # Create model
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(0.001)),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='River_Turbidity_CNN')
    
    return model, base_model

# Build model
model, base_model = create_model()
print("✅ Model architecture created!")
print(f"\n📊 Model Summary:")
model.summary()

83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
✅ Model architecture created!

📊 Model Summary:


Model: "River_Turbidity_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ xception (Functional)           │ (None, 10, 10, 2048)   │    20,861,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 21,420,714 (81.71 MB)

 Trainable params: 558,466 (2.13 MB)

 Non-trainable params: 20,862,248 (79.58 MB)

## Step 5: Setup Callbacks

In [ ]:
# Create directories for logs
MODEL_DIR = Path('./models')
LOG_DIR = Path('./logs')
MODEL_DIR.mkdir(exist_ok=True)
LOG_DIR.mkdir(exist_ok=True)

# Timestamp for unique run identification
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Callbacks
callbacks = [
    # Early stopping
    EarlyStopping(
        monitor='val_loss',
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),
    
    # Model checkpoint
    ModelCheckpoint(
        MODEL_DIR / f'xception_best_{timestamp}.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    
    # Learning rate reduction
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
    
    # TensorBoard
    TensorBoard(
        log_dir=LOG_DIR / 'tensorboard' / timestamp,
        histogram_freq=1,
        write_graph=True
    ),
    
    # CSV logger
    CSVLogger(
        LOG_DIR / f'training_history_{timestamp}.csv',
        append=True
    )
]

print("✅ Callbacks configured:")
print("   - EarlyStopping (patience=10)")
print("   - ModelCheckpoint (best model saved)")
print("   - ReduceLROnPlateau (adaptive learning rate)")
print("   - TensorBoard (real-time monitoring)")
print("   - CSVLogger (training history)")

✅ Callbacks configured:
   - EarlyStopping (patience=10)
   - ModelCheckpoint (best model saved)
   - ReduceLROnPlateau (adaptive learning rate)
   - TensorBoard (real-time monitoring)
   - CSVLogger (training history)


## Step 6: Stage 1 Training (Frozen Base)

In [6]:
# Compile model for Stage 1
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_STAGE1),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"🚀 STAGE 1: Training with FROZEN BASE MODEL")
print(f"   Epochs: {EPOCHS_STAGE1}")
print(f"   Learning Rate: {LEARNING_RATE_STAGE1}")
print(f"   Training samples: {train_generator.samples}")
print(f"   Validation samples: {val_generator.samples}\n")

# Train Stage 1
history_stage1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Stage 1 training completed!")

🚀 STAGE 1: Training with FROZEN BASE MODEL
   Epochs: 20
   Learning Rate: 0.001
   Training samples: 142
   Validation samples: 30

Epoch 1/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5022 - loss: 1.6534
Epoch 1: val_accuracy improved from None to 0.70000, saving model to models\xception_best_20251030_172444.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 16s 3s/step - accuracy: 0.5845 - loss: 1.5047 - val_accuracy: 0.7000 - val_loss: 1.2332 - learning_rate: 0.0010
Epoch 2/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7468 - loss: 1.1340
Epoch 2: val_accuracy improved from 0.70000 to 0.73333, saving model to models\xception_best_20251030_172444.keras
5/5 ━━━━━━━━━━━━━━━━━━━━ 11s 2s/step - accuracy: 0.7676 - loss: 1.1616 - val_accuracy: 0.7333 - val_loss: 1.2183 - learning_rate: 0.0010
Epoch 3/20
5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8264 - loss: 1.0426
Epoch 3: val_accuracy did not improve from 0.73333
5/5 ━━━━━━━━━━━━━━━━━━━━ 10s 2s/step - accuracy: 0.8662 - loss: 0.9749 -

KeyboardInterrupt: 

## Step 7: Stage 2 Training (Fine-Tuning)

In [ ]:
# Unfreeze base model for fine-tuning
base_model.trainable = True

# Compile model for Stage 2 with lower learning rate
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE_STAGE2),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print(f"🚀 STAGE 2: FINE-TUNING BASE MODEL")
print(f"   Epochs: {EPOCHS_STAGE2}")
print(f"   Learning Rate: {LEARNING_RATE_STAGE2}")
print(f"   Trainable layers: {sum([1 for l in model.layers if l.trainable])}\n")

# Train Stage 2
history_stage2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks,
    initial_epoch=EPOCHS_STAGE1,
    verbose=1
)

print("\n✅ Stage 2 fine-tuning completed!")

## Step 8: Save Final Model

In [ ]:
# Save final model
final_model_path = MODEL_DIR / 'xception_final.keras'
model.save(final_model_path)

print(f"✅ Final model saved: {final_model_path}")
print(f"   Model size: {final_model_path.stat().st_size / 1024 / 1024:.2f} MB")

## Step 9: Plot Training History

In [ ]:
# Combine histories from both stages
combined_history = {
    'loss': history_stage1.history['loss'] + history_stage2.history['loss'],
    'accuracy': history_stage1.history['accuracy'] + history_stage2.history['accuracy'],
    'val_loss': history_stage1.history['val_loss'] + history_stage2.history['val_loss'],
    'val_accuracy': history_stage1.history['val_accuracy'] + history_stage2.history['val_accuracy']
}

# Plot
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(combined_history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(combined_history['val_loss'], label='Val Loss', linewidth=2)
axes[0].axvline(x=EPOCHS_STAGE1, color='red', linestyle='--', label='Stage 1→2')
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(combined_history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(combined_history['val_accuracy'], label='Val Accuracy', linewidth=2)
axes[1].axvline(x=EPOCHS_STAGE1, color='red', linestyle='--', label='Stage 1→2')
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / f'training_history_{timestamp}.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Training history plot saved!")

## ✨ Summary

✅ Completed:
- Built Xception architecture with custom head
- Stage 1 training (frozen base)
- Stage 2 fine-tuning (all layers trainable)
- Saved best model and final model
- Generated training history plots

📌 **Next Steps**:
1. Move to **3_Model_Evaluation.ipynb** notebook
2. Evaluate on test set
3. Generate confusion matrix and ROC curve
4. Calculate detailed metrics